In [15]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [16]:
dias = ['domingo', 'segunda', 'terça', 'quarta', 'quinta', 'sexta', 'sábado']
n_minimo = [18,17,16,16,16,14,19]

dias_seguidos = 5
dias_folga = 2



In [17]:
model = pyo.ConcreteModel()

model.dias = pyo.RangeSet(0, len(dias)-1)
model.n_minimo = pyo.Param(model.dias, initialize=dict(zip(model.dias, n_minimo)))

model.x = pyo.Var(model.dias, bounds=(0, None), domain=pyo.NonNegativeIntegers)

In [18]:
model.n_minimo.pprint()
model.dias.pprint()

n_minimo : Size=7, Index=dias, Domain=Any, Default=None, Mutable=False
    Key : Value
      0 :    18
      1 :    17
      2 :    16
      3 :    16
      4 :    16
      5 :    14
      6 :    19
dias : Dimen=1, Size=7, Bounds=(0, 6)
    Key  : Finite : Members
    None :   True :   [0:6]


In [19]:
#objetivo

def obj_rule(model):
    return sum(model.x[d] for d in model.dias)  
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

In [20]:
# restrições
#tentando entender
#  imagina que o cara trabalha no domingo , eles trabalham 5 dias consecutivos,
#  e depois tem 2 dias de folga, ou seja, ele trabalha do domingo até quinta,
#  e folga na sexta e sábado, 
# ou seja x[0] é domingo, entao x[0] + x[1] + x[2] + x[3] + x[4] >= n_minimo[0], 
# e x[1] é segunda,entao x[1] + x[2] + x[3] + x[4] + x[5] >= n_minimo[1], e assim por diante,
#
def restricao_minimo(model, d):
    return sum(model.x[(d-k)%7] for k in range(5)) >= model.n_minimo[d]
model.restricao_minimo = pyo.Constraint(model.dias, rule=restricao_minimo)

In [21]:
model.pprint()

1 RangeSet Declarations
    dias : Dimen=1, Size=7, Bounds=(0, 6)
        Key  : Finite : Members
        None :   True :   [0:6]

1 Param Declarations
    n_minimo : Size=7, Index=dias, Domain=Any, Default=None, Mutable=False
        Key : Value
          0 :    18
          1 :    17
          2 :    16
          3 :    16
          4 :    16
          5 :    14
          6 :    19

1 Var Declarations
    x : Size=7, Index=dias
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :  None :  None : False :  True : NonNegativeIntegers
          1 :     0 :  None :  None : False :  True : NonNegativeIntegers
          2 :     0 :  None :  None : False :  True : NonNegativeIntegers
          3 :     0 :  None :  None : False :  True : NonNegativeIntegers
          4 :     0 :  None :  None : False :  True : NonNegativeIntegers
          5 :     0 :  None :  None : False :  True : NonNegativeIntegers
          6 :     0 :  None :  None : False :  True : NonNega

In [25]:
opt = SolverFactory('cplex')
res =   opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpl1dwz1j0.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmptkq2u87b.pyomo.lp' read.
Read time = 0.03 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmptkq2u87b.pyomo.lp
Objective sense      : Minimize
Variables            :       7  [General Integer: 7]
Objective nonzeros   :       7
Linear constraints   :       7  [Greater: 7]
  Nonzeros           :      35
  RHS nonzeros       :       7

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1.000000    

In [26]:
for x in model.x:
    print(f'{dias[x]}: {model.x[x].value}'  )

domingo: 3.0
segunda: 0.0
terça: 5.0
quarta: 2.0
quinta: 6.0
sexta: 2.0
sábado: 6.0
